# Assignment 3 - Carbon-Efficient RAG (Config A vs Config B)
**Before you start:** Runtime > Change runtime type > T4 GPU. Upload `full_text.pdf` using the folder icon on the left.
Run the cells in order. Keep `TRIAL_MODE = True` the first time (1 question per config). Then set it to `False` and run again for the real 20 runs.

## Cell 1 - Install (run once, then Runtime > Restart session if Colab asks)

In [ ]:
!pip install -q transformers accelerate sentence-transformers pypdf codecarbon

## Cell 2 - Setup (edit only the first block of settings)

In [ ]:
# ===== SETTINGS YOU MAY EDIT =====
TRIAL_MODE = False          # True = 1 question per config (test). False = all 10 questions (real run)
USE_OFFLINE_TRACKER = False  # set True if CodeCarbon gives a network/location error
COUNTRY_ISO = "CAN"
PDF_PATH = "full_text.pdf"
GEN_MODEL = "Qwen/Qwen2.5-3B-Instruct"   # same generator for A and B
MAX_NEW_TOKENS = 150
# =================================

import os, time, glob, gc
import numpy as np, pandas as pd, torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from codecarbon import EmissionsTracker, OfflineEmissionsTracker

FILE_PREFIX = "a3trial" if TRIAL_MODE else "a3results"

CONFIGS = {
    "A": dict(embed="sentence-transformers/all-MiniLM-L6-v2", chunk=256, overlap=50, k=8, qprefix=""),
    "B": dict(embed="BAAI/bge-small-en-v1.5", chunk=512, overlap=50, k=3,
              qprefix="Represent this sentence for searching relevant passages: "),
}

# (question, reference answer, page in the PDF) - for YOUR manual grading
QUESTIONS = [
    ("How many survey responses did the system record in total before cleaning?", "111", 12),
    ("How many responses were retained after pre-processing?", "108", 13),
    ("Responses completed in less than how long were flagged as likely inattentive?", "Less than 1 minute", 12),
    ("Which method did the authors use to fill in missing survey data?", "Multiple Imputation by Chained Equations (MICE), using Predictive Mean Matching", 12),
    ("How many items does Colquitt's fairness-perception questionnaire have?", "20", 7),
    ("Which sampling technique did the authors use to recruit respondents, and through which channels?", "Purposive sampling; LinkedIn, social media, and professional events", 11),
    ("Which R package was used for the ordinal logistic regression?", "MASS", 17),
    ("Which linkage method was used for the hierarchical clustering of job satisfaction factors?", "Average linkage", 15),
    ("How many composite factors resulted from grouping the job satisfaction factors?", "Seven", 16),
    ("Which interpersonal fairness item had the greatest effect on overall job satisfaction, and what was its odds ratio?", "Treated with respect (Int2), OR = 3.40", 22),
]

def make_tracker(name):
    kw = dict(project_name=name, measure_power_secs=1, save_to_file=False,
              log_level="error", allow_multiple_runs=True)
    if USE_OFFLINE_TRACKER:
        return OfflineEmissionsTracker(country_iso_code=COUNTRY_ISO, **kw)
    return EmissionsTracker(**kw)

# ---- Load the paper ----
reader = PdfReader(PDF_PATH)
TEXT = "\n".join((p.extract_text() or "") for p in reader.pages)
print("Pages:", len(reader.pages), "| characters:", len(TEXT))

# ---- Load the generator (same for A and B) ----
tok = AutoTokenizer.from_pretrained(GEN_MODEL)
gen = AutoModelForCausalLM.from_pretrained(GEN_MODEL, torch_dtype=torch.float16, device_map="cuda")
gen.eval()

def n_tokens(s):
    return len(tok(s, add_special_tokens=False)["input_ids"])

def chunk_text(text, size, overlap):
    ids = tok(text, add_special_tokens=False)["input_ids"]
    step = size - overlap
    out = []
    for s in range(0, len(ids), step):
        out.append(tok.decode(ids[s:s + size]))
        if s + size >= len(ids):
            break
    return out

def build_prompt(question, ctx_chunks):
    context = "\n\n".join(f"[Passage {i+1}]\n{c}" for i, c in enumerate(ctx_chunks))
    sys_msg = ("Answer the question using ONLY the passages provided. Be short and direct. "
               "If the answer is not in the passages, say: Not found in the document.")
    user_msg = f"Passages:\n{context}\n\nQuestion: {question}"
    msgs = [{"role": "system", "content": sys_msg}, {"role": "user", "content": user_msg}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def generate(prompt):
    inputs = tok(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = gen.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                           pad_token_id=tok.eos_token_id, max_time=120)
    new_ids = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(new_ids, skip_special_tokens=True).strip(), inputs["input_ids"].shape[1], len(new_ids)

def run_config(name):
    out_file = f"{FILE_PREFIX}_{name}.csv"
    if os.path.exists(out_file):
        print(f"{out_file} already exists - skipping config {name}.")
        return
    cfg = CONFIGS[name]
    chunks = chunk_text(TEXT, cfg["chunk"], cfg["overlap"])
    print(f"Config {name}: {len(chunks)} chunks of {cfg['chunk']} tokens")
    emb = SentenceTransformer(cfg["embed"], device="cuda")
    emb.max_seq_length = 512   # same limit for both embedding models (no silent truncation)
    chunk_vecs = emb.encode(chunks, normalize_embeddings=True, batch_size=64, convert_to_numpy=True)

    def one_query(q):
        t0 = time.perf_counter()
        qv = emb.encode([cfg["qprefix"] + q], normalize_embeddings=True, convert_to_numpy=True)[0]
        top = np.argsort(-(chunk_vecs @ qv))[:cfg["k"]]
        t_ret = (time.perf_counter() - t0) * 1000
        ctx = [chunks[i] for i in top]
        prompt = build_prompt(q, ctx)
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        ans, p_tok, o_tok = generate(prompt)
        torch.cuda.synchronize()
        t_gen = (time.perf_counter() - t1) * 1000
        t_in = n_tokens(q) + sum(n_tokens(c) for c in ctx)
        return t_ret, t_gen, t_in, p_tok, o_tok, ans, ctx

    # warm-up (not saved)
    tr = make_tracker("warmup"); tr.start(); one_query(QUESTIONS[0][0]); tr.stop()

    rows = []
    for qi, (q, ref, page) in enumerate(QUESTIONS, 1):
        if TRIAL_MODE and qi > 1:
            break
        tr = make_tracker(f"A3_{name}_Q{qi}")
        tr.start()
        t_ret, t_gen, t_in, p_tok, o_tok, ans, ctx = one_query(q)
        em = tr.stop()
        co2_g = float(em) * 1000 if em is not None else float("nan")
        rows.append(dict(config=name, q_id=qi, question=q, reference_answer=ref, ref_page=page,
                         embed_model=cfg["embed"], chunk_size=cfg["chunk"], overlap=cfg["overlap"], k=cfg["k"],
                         T_retriever_ms=round(t_ret, 2), T_generator_ms=round(t_gen, 2),
                         T_total_ms=round(t_ret + t_gen, 2), T_in_tokens=t_in,
                         prompt_tokens_full=p_tok, output_tokens=o_tok, co2e_g=co2_g,
                         answer=ans, retrieved_chunks=" ||| ".join(ctx), pass_fail=""))
        print(f"[{name}] Q{qi}: ret {t_ret:.0f} ms | gen {t_gen:.0f} ms | T_in {t_in} | CO2e {co2_g:.6f} g\n   -> {ans[:120]}")
    pd.DataFrame(rows).to_csv(out_file, index=False)
    print("Saved", out_file)
    del emb, chunk_vecs
    gc.collect(); torch.cuda.empty_cache()


Pages: 44 | characters: 170399


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

## Cell 3 - Run Config A (baseline: MiniLM, 256 tokens, k=8)

In [ ]:
run_config("A")

Config A: 200 chunks of 256 tokens


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[A] Q1: ret 7 ms | gen 759 ms | T_in 2061 | CO2e 0.005111 g
   -> 111
[A] Q2: ret 11 ms | gen 1056 ms | T_in 2057 | CO2e 0.006609 g
   -> 108 responses were retained after pre-processing.
[A] Q3: ret 8 ms | gen 1370 ms | T_in 2064 | CO2e 0.008940 g
   -> Responses completed in less than 1 minute were flagged as likely inattentive.
[A] Q4: ret 10 ms | gen 1259 ms | T_in 2061 | CO2e 0.008499 g
   -> The authors used Multiple Imputation by Chained Equations (MICE).
[A] Q5: ret 8 ms | gen 1212 ms | T_in 2062 | CO2e 0.008232 g
   -> Colquitt's fairness-perception questionnaire has 20 items.
[A] Q6: ret 8 ms | gen 1682 ms | T_in 2064 | CO2e 0.010625 g
   -> The authors used purposive sampling to recruit software practitioners from diverse backgrounds. They utilized recruitmen
[A] Q7: ret 14 ms | gen 922 ms | T_in 2059 | CO2e 0.006694 g
   -> Not found in the document
[A] Q8: ret 10 ms | gen 1511 ms | T_in 2062 | CO2e 0.010538 g
   -> The average linkage method was used for the hierarchical c

## Cell 4 - Run Config B (optimized: bge-small, 512 tokens, k=3)

In [ ]:
run_config("B")

Config B: 90 chunks of 512 tokens


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[B] Q1: ret 12 ms | gen 582 ms | T_in 1549 | CO2e 0.003953 g
   -> 111
[B] Q2: ret 16 ms | gen 587 ms | T_in 1545 | CO2e 0.003963 g
   -> 108
[B] Q3: ret 14 ms | gen 1407 ms | T_in 1552 | CO2e 0.009930 g
   -> Responses completed in less than 1 minute were flagged as likely inattentive.
[B] Q4: ret 16 ms | gen 1840 ms | T_in 1549 | CO2e 0.012491 g
   -> The authors used Multiple Imputation by Chained Equations (MICE) with Predictive Mean Matching to fill in missing data.
[B] Q5: ret 12 ms | gen 1057 ms | T_in 1550 | CO2e 0.007093 g
   -> Colquitt's fairness-perception questionnaire has 20 items.
[B] Q6: ret 12 ms | gen 1431 ms | T_in 1552 | CO2e 0.009241 g
   -> The authors used purposive sampling to recruit respondents. They reached out via LinkedIn, social media, and professiona
[B] Q7: ret 15 ms | gen 676 ms | T_in 1547 | CO2e 0.004542 g
   -> Not found in the document
[B] Q8: ret 12 ms | gen 1077 ms | T_in 1550 | CO2e 0.007053 g
   -> The average linkage method was used for the hie

## Cell 5 - Merge A + B into one CSV and download it

In [ ]:
files_found = sorted(glob.glob(f"{FILE_PREFIX}_[AB].csv"))
print("Found:", files_found)
all_df = pd.concat([pd.read_csv(f) for f in files_found], ignore_index=True)
all_df.to_csv(f"{FILE_PREFIX}_all.csv", index=False)
display(all_df[["config","q_id","T_retriever_ms","T_generator_ms","T_total_ms","T_in_tokens","co2e_g","answer"]])
from google.colab import files
files.download(f"{FILE_PREFIX}_all.csv")

Found: ['a3results_A.csv', 'a3results_B.csv']


,config,q_id,T_retriever_ms,T_generator_ms,T_total_ms,T_in_tokens,co2e_g,answer
0,A,1,7.49,758.94,766.43,2061,0.005111,111
1,A,2,11.36,1055.92,1067.28,2057,0.006609,108 responses were retained after pre-processing.
2,A,3,8.33,1370.14,1378.47,2064,0.008940,Responses completed in less than 1 minute were...
3,A,4,9.75,1258.71,1268.46,2061,0.008499,The authors used Multiple Imputation by Chaine...
4,A,5,7.79,1211.79,1219.57,2062,0.008232,Colquitt's fairness-perception questionnaire h...
5,A,6,7.85,1682.05,1689.90,2064,0.010625,The authors used purposive sampling to recruit...
6,A,7,14.21,922.16,936.37,2059,0.006694,Not found in the document
7,A,8,9.66,1511.19,1520.85,2062,0.010538,The average linkage method was used for the hi...
8,A,9,9.50,691.42,700.93,2060,0.004648,seven
9,A,10,7.95,2146.40,2154.35,2068,0.013881,Being treated with respect (Int2) had the grea...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 6 - AFTER you grade (only for the real run)
1. Open `a3results_all.csv`, fill the `pass_fail` column with `Pass` or `Fail` (correct AND grounded in the paper), save it.
2. Upload it to Colab as `a3results_graded.csv`, then run this cell to get the report table.

In [ ]:
g = pd.read_csv("a3results_graded.csv")
g["acc"] = (g["pass_fail"].str.strip().str.lower() == "pass").astype(int)
s = g.groupby("config").agg(T_retriever_ms=("T_retriever_ms","mean"), T_generator_ms=("T_generator_ms","mean"),
                            T_total_ms=("T_total_ms","mean"), T_in_tokens=("T_in_tokens","mean"),
                            co2e_g=("co2e_g","mean"), accuracy=("acc","mean")).round(4)
s["accuracy_per_total_second"] = (s["accuracy"] / (s["T_total_ms"] / 1000)).round(4)
s["accuracy_per_mg_CO2e"] = (s["accuracy"] / (s["co2e_g"] * 1000)).round(4)
display(s)
s.to_csv("a3_summary_table.csv")
from google.colab import files
files.download("a3_summary_table.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'a3results_graded.csv'

## Cell 7 - Optional: free the GPU when completely finished (download your files first!)

In [ ]:
from google.colab import runtime
runtime.unassign()

In [ ]:
from google.colab import files
up = files.upload()
import os
os.rename(list(up.keys())[0], "full_text.pdf")
print(os.listdir("."))

Saving full_text.pdf to full_text.pdf
['.config', 'full_text.pdf', 'sample_data']
